In [15]:
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from dotenv import load_dotenv
import os

import operator

load_dotenv()

True

In [4]:
model = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite-preview",   
    google_api_key=os.getenv('GEMINI_API_KEY')
)

In [5]:
class EvaluationSchema(BaseModel):
    feedback: str = Field(description='detailed feedback for essay')
    score: int = Field(description='Score out of 10', ge=0, le=10)

In [9]:
structured_model = model.with_structured_output(EvaluationSchema)

In [17]:
class EssayState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[int], operator.add]
    avg_score: float

In [23]:
def evaluate_language(state: EssayState):
    essay = state['essay']
    prompt = f'Evaluate the language quality of the essay and give it a score out of 10\n {essay}'

    output = structured_model.invoke(prompt)
    return {'language_feedback': output.feedback, 'individual_scores': [output.score]}

In [24]:
def evaluate_analysis(state: EssayState):
    essay = state['essay']
    prompt = f'Evaluate the analysis quality of the essay and give it a score out of 10\n {essay}'

    output = structured_model.invoke(prompt)
    return {'analysis_feedback': output.feedback, 'individual_scores': [output.score]}

In [25]:
def evaluate_thought(state: EssayState):
    essay = state['essay']
    prompt = f'Evaluate the clarity of thought quality of the essay and give it a score out of 10\n {essay}'

    output = structured_model.invoke(prompt)
    return {'thought_feedback': output.feedback, 'individual_scores': [output.score]}

In [26]:
def final_evaluation(state: EssayState):

    # summary feedback
    prompt = f'Based on the following feedbacks, create a summarized feedback:\n Analysis feedback: {state["analysis_feedback"]} \nThought of Clarity Feedback: {state["clarity_feedback"]}\n Language Feedback: {state["language_feedback"]}'
    final_result = model.invoke(prompt).content
    state["overall_feedback"] = final_result[0]['text']

    #avg result
    state['avg_score'] = sum(state["individual_scores"])/len(state["individual_scores"])

    return state

In [ ]:
graph = StateGraph(EssayState)

graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_thought', evaluate_thought)
graph.add_node('final_evaluation', final_evaluation)

graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_thought')

graph.add_edge('evaluate_language', END)
graph.add_edge('evaluate_analysis', END)
graph.add_edge('evaluate_thought', END)




